# Google Colab — Nuclear Reset & Full Run

**Repo:** https://github.com/hello-umang/PM25_Pollution_Forecast

## How to open this notebook in Colab
1. Go to [Google Colab](https://colab.research.google.com/)
2. **File → Open notebook → GitHub**
3. Paste: `hello-umang/PM25_Pollution_Forecast`
4. Open: `notebooks/Colab_Nuclear_Reset.ipynb`
5. Runtime → **Run all**

Or open directly (after push):
https://colab.research.google.com/github/hello-umang/PM25_Pollution_Forecast/blob/main/notebooks/Colab_Nuclear_Reset.ipynb

## What the cell below does
1. Deletes any old `/content/PM25_Pollution_Forecast` folder
2. Fresh `git clone` from GitHub
3. Installs light dependencies (no XGBoost)
4. Regenerates sample data
5. Trains models + builds visuals
6. Prints full process log + comparison table


## One-cell nuclear reset (run this)

In [ ]:
# ============================================================
# ONE-CELL NUCLEAR RESET — Google Colab / class demo
# Deletes old project folder, reclones GitHub, regenerates,
# trains light sklearn models, prints full process log.
# No XGBoost / LightGBM / CatBoost.
# ============================================================

import os
import shutil
import subprocess
import sys
import time
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Image

def log(msg: str) -> None:
    print(f"[LOG] {msg}")

print("=" * 60)
log("FULL RESET START")
print("=" * 60)

# ---------- 1) Delete old project ----------
os.chdir("/content")
project = Path("/content/PM25_Pollution_Forecast")
zip_path = Path("/content/PM25_Pollution_Forecast.zip")

if project.exists():
    log(f"Deleting old folder: {project}")
    shutil.rmtree(project, ignore_errors=True)
    log(f"Deleted folder | still_exists={project.exists()}")
else:
    log("No old PM25_Pollution_Forecast folder found")

if zip_path.exists():
    log(f"Deleting old zip: {zip_path}")
    zip_path.unlink(missing_ok=True)
    log("Old zip deleted")

# ---------- 2) Clone fresh from GitHub ----------
repo = "https://github.com/hello-umang/PM25_Pollution_Forecast.git"
log(f"Cloning: {repo}")
clone = subprocess.run(
    ["git", "clone", repo],
    capture_output=True,
    text=True,
)
print("[LOG] git stdout:\n", clone.stdout)
print("[LOG] git stderr:\n", clone.stderr)
if clone.returncode != 0:
    raise RuntimeError(f"git clone failed with code {clone.returncode}")

assert project.exists(), "Clone failed — project folder missing"
os.chdir(project)
log(f"cwd = {Path.cwd()}")

log("Top-level files/folders after clone:")
for p in sorted(project.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"  {kind}  {p.name}")

# ---------- 3) Install dependencies ----------
log("Installing packages (light stack, no XGBoost)...")
pip = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "matplotlib",
        "seaborn",
        "scikit-learn",
        "joblib",
        "scipy",
        "plotly",
        "nbformat",
    ],
    capture_output=True,
    text=True,
)
if pip.returncode != 0:
    print(pip.stdout)
    print(pip.stderr)
    raise RuntimeError("pip install failed")

import numpy
import sklearn

log(f"numpy={numpy.__version__}  sklearn={sklearn.__version__}")

# ---------- 4) Clean generated artifacts (force full regen) ----------
log("Cleaning generated data/models/visuals for full regenerate...")
for rel in [
    "data/pm25_raw.csv",
    "data/pm25_cleaned_features.csv",
]:
    fp = project / rel
    if fp.exists():
        fp.unlink()
        log(f"removed file {rel}")

for dname in ["models", "visuals", "data"]:
    d = project / dname
    if d.exists() and dname in {"models", "visuals"}:
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    log(f"ready dir: {dname}/")

# ---------- 5) Regenerate data ----------
log("Running generate_data.py ...")
t0 = time.time()
g = subprocess.run([sys.executable, "generate_data.py"], capture_output=True, text=True)
print(g.stdout)
if g.stderr:
    print("[LOG][stderr]", g.stderr)
if g.returncode != 0:
    raise RuntimeError("generate_data.py failed")
log(f"generate_data.py done in {time.time() - t0:.1f}s")

# ---------- 6) Train + EDA + evaluate ----------
log("Running eda_and_model.py (train/evaluate) ...")
t1 = time.time()
m = subprocess.run([sys.executable, "eda_and_model.py"], capture_output=True, text=True)
print(m.stdout)
if m.stderr:
    print("[LOG][stderr]", m.stderr)
if m.returncode != 0:
    raise RuntimeError("eda_and_model.py failed")
log(f"eda_and_model.py done in {time.time() - t1:.1f}s")

# ---------- 7) Results log ----------
print("=" * 60)
log("FINAL RESULTS")
print("=" * 60)

summary_path = project / "models" / "summary.json"
summary = json.loads(summary_path.read_text())
log(f"BEST MODEL = {summary['best_model']}")
log(f"METRICS    = {summary['best_metrics']}")
log(f"MODELS    = {summary['models_trained']}")
log(f"CLEANING   = {summary['cleaning_log']}")

print("\n[LOG] MODEL COMPARISON TABLE")
cmp = pd.read_csv(project / "models" / "model_comparison.csv")
display(cmp)

print("\n[LOG] Regenerated files:")
for folder in ["data", "models", "visuals"]:
    print(f"  {folder}/")
    for f in sorted((project / folder).iterdir()):
        print(f"    - {f.name:40s} {f.stat().st_size/1024:8.1f} KB")

print("\n[LOG] KEY CHARTS")
for img in [
    "visuals/daily_pm25_trend.png",
    "visuals/model_comparison_rmse.png",
    "visuals/actual_vs_predicted.png",
    "visuals/feature_importance.png",
    "visuals/error_analysis.png",
]:
    path = project / img
    if path.exists():
        print("\n", img)
        display(Image(filename=str(path)))
    else:
        log(f"missing chart: {img}")

log("FULL RESET COMPLETE")
print("=" * 60)


## Submit this notebook on GitHub

This file already belongs in the repo at:

```text
notebooks/Colab_Nuclear_Reset.ipynb
```

### If you still need to push it from your laptop

```bash
cd PM25_Pollution_Forecast
git add notebooks/Colab_Nuclear_Reset.ipynb README.md
git commit -m "Add Colab one-cell nuclear reset notebook for class demo"
git push origin main
```

### Class submission links
- Repo: https://github.com/hello-umang/PM25_Pollution_Forecast
- Colab open-from-GitHub:  
  https://colab.research.google.com/github/hello-umang/PM25_Pollution_Forecast/blob/main/notebooks/Colab_Nuclear_Reset.ipynb
